# Dynamic vs Fixed/Variable Electricity Cost Comparison
Interactive dashboard comparing hourly dynamic spot-price costs with fixed/variable contract costs

## 1. Import Required Libraries and Load Data

In [88]:
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
from datetime import datetime, timedelta
import pyarrow.parquet as pq

warnings.filterwarnings('ignore')

# Constants
VAT_RATE = 0.21
ELEC_TAX = 0.11085  # €/kWh
PIEK_HOURS = list(range(7, 23))  # 07:00 - 23:00
DAL_HOURS = list(range(23, 24)) + list(range(0, 7))  # 23:00 - 07:00

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 2. Load and Prepare ENTSOE Spot Price Data

In [89]:
# Load ENTSOE spot prices
entsoe_parquet_path = r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Visualizations\entsoe_prices_NL_2025_2026_combined.parquet'

try:
    # Check if file exists first
    import os
    if not os.path.exists(entsoe_parquet_path):
        print(f"✗ File not found: {entsoe_parquet_path}")
        df_entsoe = None
    else:
        df_entsoe = pd.read_parquet(entsoe_parquet_path)
        print(f"✓ ENTSOE data loaded: {len(df_entsoe)} rows")
        print(f"  Columns: {df_entsoe.columns.tolist()}")
        print(f"  Index type: {type(df_entsoe.index)}")
        
        # Identify timestamp and price columns
        ts_col = None
        price_col = None
        
        # Try common column names
        for col in ['ts_utc', 'timestamp', 'time', 'date']:
            if col in df_entsoe.columns:
                ts_col = col
                break
        
        for col in ['price', 'Price', 'price_eur_per_mwh', 'price_eur_per_kwh']:
            if col in df_entsoe.columns:
                price_col = col
                break
        
        if ts_col is None or price_col is None:
            print(f"✗ Could not identify timestamp or price columns")
            print(f"  Looking for: timestamp column in {['ts_utc', 'timestamp', 'time', 'date']}")
            print(f"  Looking for: price column in {['price', 'Price', 'price_eur_per_mwh', 'price_eur_per_kwh']}")
            df_entsoe = None
        else:
            # Set proper index
            df_entsoe[ts_col] = pd.to_datetime(df_entsoe[ts_col])
            df_entsoe = df_entsoe.set_index(ts_col)
            
            print(f"  Using timestamp column: {ts_col}")
            print(f"  Using price column: {price_col}")
            print(f"  Date range: {df_entsoe.index.min()} to {df_entsoe.index.max()}")
            
            # Rename or create price_eur_per_kwh column
            # Assuming price column is already in €/kWh or is in €/MWh
            if 'price_eur_per_mwh' in df_entsoe.columns or (price_col == 'price' and df_entsoe[price_col].max() > 100):
                # Likely in €/MWh, convert to €/kWh
                df_entsoe['price_eur_per_kwh'] = df_entsoe[price_col] / 1000
                print("  Converted price from €/MWh to €/kWh")
            else:
                # Already in €/kWh
                df_entsoe['price_eur_per_kwh'] = df_entsoe[price_col]
                print("  Price already in €/kWh")
            
            print(f"✓ Spot prices ready")
            print(f"  Price range: €{df_entsoe['price_eur_per_kwh'].min():.4f} - €{df_entsoe['price_eur_per_kwh'].max():.4f}/kWh")
    
except Exception as e:
    import traceback
    print(f"✗ Error loading ENTSOE data: {e}")
    print(f"  Traceback: {traceback.format_exc()}")
    df_entsoe = None

✓ ENTSOE data loaded: 38388 rows
  Columns: ['ts_utc', 'price']
  Index type: <class 'pandas.core.indexes.range.RangeIndex'>
  Using timestamp column: ts_utc
  Using price column: price
  Date range: 2025-01-01 00:15:00+00:00 to 2026-02-04 23:45:00+00:00
  Price already in €/kWh
✓ Spot prices ready
  Price range: €-0.3500 - €0.5235/kWh


## 3. Load Provider Tariffs and Calculate Dynamic Costs

In [90]:
import sys
from pathlib import Path
sys.path.insert(0, r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Visualizations')

try:
    from load_tariffs import load_provider_tariffs, normalize_name
    print("✓ load_tariffs module imported")
except ImportError as e:
    print(f"✗ Could not import load_tariffs: {e}")

# Load provider tariffs
tariff_xlsx_path = Path(r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Visualizations\provider_tariffs.xlsx')

try:
    if not tariff_xlsx_path.exists():
        print(f"✗ Provider tariffs file not found: {tariff_xlsx_path}")
        dynamic_providers = {}
    else:
        tariff_book = load_provider_tariffs(tariff_xlsx_path)
        dynamic_providers = {}
        
        for provider_key, contracts in tariff_book.providers.items():
            for contract_type, tariff in contracts.items():
                if contract_type == 'dynamic':
                    if provider_key not in dynamic_providers:
                        dynamic_providers[provider_key] = {}
                    dynamic_providers[provider_key]['margin'] = tariff.price_eur_per_kwh
                    dynamic_providers[provider_key]['provider_name'] = tariff.provider
        
        print(f"✓ Provider tariffs loaded: {len(dynamic_providers)} providers with dynamic contracts")
        for provider_key, data in list(dynamic_providers.items())[:5]:
            print(f"  {data['provider_name']}: €{data['margin']:.6f}/kWh margin")
        
except Exception as e:
    import traceback
    print(f"✗ Error loading provider tariffs: {e}")
    print(f"  Traceback: {traceback.format_exc()}")
    dynamic_providers = {}

✓ load_tariffs module imported
✓ Provider tariffs loaded: 19 providers with dynamic contracts
  Mega Energie: €0.018150/kWh margin
  Greenchoice: €0.033880/kWh margin
  Vandebron: €0.025710/kWh margin
  Budget Energie: €0.020990/kWh margin
  Oxxio: €0.018490/kWh margin


In [91]:
# Calculate dynamic costs for each provider
def calculate_dynamic_costs(df_entsoe, dynamic_providers):
    """Calculate hourly dynamic costs per provider"""
    if df_entsoe is None:
        print("✗ Cannot calculate dynamic costs: ENTSOE data is None")
        return {}
    
    if not dynamic_providers:
        print("✗ No dynamic providers found")
        return {}
    
    dynamic_costs = {}
    
    for provider_key, provider_data in dynamic_providers.items():
        provider_name = provider_data['provider_name']
        margin = provider_data['margin']
        
        # Formula: (spot_price * (1 + VAT_RATE) + margin + elec_tax)
        cost_per_kwh = (df_entsoe['price_eur_per_kwh'] * (1 + VAT_RATE) + margin + ELEC_TAX)
        
        df_provider = pd.DataFrame({
            'timestamp': df_entsoe.index,
            'spot_price_eur_per_kwh': df_entsoe['price_eur_per_kwh'],
            'total_cost_eur_per_kwh': cost_per_kwh,
            'provider': provider_name,
            'margin': margin
        })
        
        dynamic_costs[provider_name] = df_provider
    
    return dynamic_costs

dynamic_costs = calculate_dynamic_costs(df_entsoe, dynamic_providers)

if not dynamic_costs:
    print("✗ No dynamic costs were calculated. Check:")
    print("  - ENTSOE data loaded (Cell 2 status)")
    print("  - Provider tariffs loaded with 'dynamic' contracts (Cell 3 status)")
else:
    print(f"✓ Dynamic costs calculated for {len(dynamic_costs)} providers")

    if dynamic_costs:
        first_provider = list(dynamic_costs.keys())[0]
        print(f"  Example ({first_provider}):")
        print(f"    Min cost: €{dynamic_costs[first_provider]['total_cost_eur_per_kwh'].min():.4f}/kWh")
        print(f"    Max cost: €{dynamic_costs[first_provider]['total_cost_eur_per_kwh'].max():.4f}/kWh")
        print(f"    Avg cost: €{dynamic_costs[first_provider]['total_cost_eur_per_kwh'].mean():.4f}/kWh")

✓ Dynamic costs calculated for 19 providers
  Example (Mega Energie):
    Min cost: €-0.2945/kWh
    Max cost: €0.7624/kWh
    Avg cost: €0.2367/kWh


## 4. Load Fixed/Variable Contract Data

In [92]:
# Load monthly contract data from JSON files (same as tariff_interactive.ipynb)
import json
from pathlib import Path

months_data = {
    'February 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_februari 2025.json',
    'March 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_maart 2025.json',
    'April 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_april 2025.json',
    'May 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_mei 2025.json',
    'June 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_juni 2025.json',
    'July 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_juli 2025.json',
    'August 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_augustus 2025.json',
    'September 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_september 2025.json',
    'October 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_oktober 2025.json',
    'November 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_november 2025.json',
    'December 2025': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2025\contracts_december 2025.json',
    'January 2026': r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\2026\contracts_januari 2026.json',
}

# Load all monthly data
df_fixed_monthly_list = []
for month, path in months_data.items():
    try:
        with open(path, 'r', encoding='utf-8') as f:
            month_data = json.load(f)
        
        for key, contract_info in month_data.items():
            parts = key.split(' | ')
            provider = parts[0] if len(parts) > 0 else 'Unknown'
            
            row = {
                'month': month,
                'provider': provider,
                'contract_name': contract_info.get('Contractnaam', ''),
                'contract_type': contract_info.get('Contractduur', ''),
                'piek_tariff': float(str(contract_info.get('Variabel elektriciteit enkel / piek per kWh', 'NaN')).replace(',', '.')),
                'dal_tariff': float(str(contract_info.get('Variabel elektriciteit dal per kWh', 'NaN')).replace(',', '.')) \
                    if contract_info.get('Variabel elektriciteit dal per kWh', 'NaN') != 'NaN' else np.nan,
            }
            df_fixed_monthly_list.append(row)
    except Exception as e:
        print(f"  Warning loading {month}: {e}")

df_fixed_monthly = pd.DataFrame(df_fixed_monthly_list)
print(f"✓ Fixed/variable contract data loaded: {len(df_fixed_monthly)} records across 12 months")
print(f"  Providers: {df_fixed_monthly['provider'].nunique()}")
print(f"  Months: {df_fixed_monthly['month'].nunique()}")

✓ Fixed/variable contract data loaded: 2502 records across 12 months
  Providers: 47
  Months: 12


## 5. Persistent Parquet Cache & Resolution-Aware Data Retrieval

In [93]:
from pathlib import Path

cache_dir = Path(r'c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Visualizations\cache')
cache_dir.mkdir(exist_ok=True)

def load_or_build_parquet(path, build_fn):
    """Load parquet from disk or build + save if missing"""
    if path.exists():
        return pd.read_parquet(path)
    else:
        df = build_fn()
        df.to_parquet(path, index=False)
        return df

def build_spot_15m():
    df = df_entsoe[['price_eur_per_kwh']].copy()
    df = df.reset_index()
    df.columns = ['datetime', 'spot_eur_per_kwh']
    df['datetime'] = pd.to_datetime(df['datetime']).dt.tz_convert('Europe/Amsterdam')
    return df

def build_spot_hourly():
    df = build_spot_15m()
    # Use strftime to avoid DST ambiguity in floor
    df['datetime_str'] = df['datetime'].dt.strftime('%Y-%m-%d %H:00:00')
    df['datetime_hour'] = pd.to_datetime(df['datetime_str']).dt.tz_localize('Europe/Amsterdam', ambiguous='NaT')
    df = df.dropna(subset=['datetime_hour'])
    agg = df.groupby('datetime_hour')['spot_eur_per_kwh'].mean().reset_index()
    agg.columns = ['datetime_hour', 'spot_hourly_mean_eur_per_kwh']
    return agg

def build_spot_daily():
    hourly_df = load_or_build_parquet(cache_dir / 'spot_hourly.parquet', build_spot_hourly)
    hourly_df['date'] = pd.to_datetime(hourly_df['datetime_hour']).dt.date
    agg = hourly_df.groupby('date')['spot_hourly_mean_eur_per_kwh'].mean().reset_index()
    agg.columns = ['date', 'spot_daily_mean_eur_per_kwh']
    return agg

def build_fv_segments():
    rows = []
    for idx, row in df_fixed_monthly.iterrows():
        month_str = row['month']
        month_date = pd.to_datetime(month_str)
        year, month_num = month_date.year, month_date.month
        month_start = pd.Timestamp(f'{year}-{month_num:02d}-01', tz='Europe/Amsterdam')
        if month_num == 12:
            month_end = pd.Timestamp(f'{year+1}-01-01', tz='Europe/Amsterdam')
        else:
            month_end = pd.Timestamp(f'{year}-{month_num+1:02d}-01', tz='Europe/Amsterdam')
        rows.append({
            'provider': row['provider'],
            'contract_name': row['contract_name'],
            'contract_type': row['contract_type'],
            'month_start': month_start,
            'month_end': month_end,
            'piek_price_eur_per_kwh': row['piek_tariff'],
            'dal_price_eur_per_kwh': row['dal_tariff']
        })
    return pd.DataFrame(rows)

def build_margin_segments():
    rows = []
    for provider, data in dynamic_providers.items():
        rows.append({
            'provider': data['provider_name'],
            'valid_from': pd.Timestamp('2025-01-01', tz='Europe/Amsterdam'),
            'valid_to': pd.Timestamp('2026-12-31', tz='Europe/Amsterdam'),
            'margin_eur_per_kwh': data['margin']
        })
    return pd.DataFrame(rows)

def build_energy_tax():
    return pd.DataFrame({
        'year': [2025, 2026],
        'energy_tax_eur_per_kwh': [ELEC_TAX, ELEC_TAX]
    })

spot_15m = load_or_build_parquet(cache_dir / 'spot_15m.parquet', build_spot_15m)
spot_hourly = load_or_build_parquet(cache_dir / 'spot_hourly.parquet', build_spot_hourly)
spot_daily = load_or_build_parquet(cache_dir / 'spot_daily.parquet', build_spot_daily)
fv_segments = load_or_build_parquet(cache_dir / 'fv_month_segments.parquet', build_fv_segments)
margin_segments = load_or_build_parquet(cache_dir / 'margin_segments.parquet', build_margin_segments)
energy_tax_year = load_or_build_parquet(cache_dir / 'energy_tax_year.parquet', build_energy_tax)

print("✓ Cache infrastructure ready")
print(f"  Spot 15m: {len(spot_15m)} rows")
print(f"  Spot hourly: {len(spot_hourly)} rows")
print(f"  Spot daily: {len(spot_daily)} rows")
print(f"  Fixed/variable segments: {len(fv_segments)} rows")
print(f"  Margin segments: {len(margin_segments)} rows")

✓ Cache infrastructure ready
  Spot 15m: 38388 rows
  Spot hourly: 9596 rows
  Spot daily: 401 rows
  Fixed/variable segments: 2502 rows
  Margin segments: 19 rows


In [94]:
series_cache = {}

def get_series(contract_type, provider, contract_name, start, end, resolution):
    """
    Retrieve cost timeseries at specified resolution for selected window only.
    Caches results to avoid recomputation. Never upsamples globally.
    """
    cache_key = (contract_type, provider, contract_name, str(start), str(end), resolution)
    if cache_key in series_cache:
        return series_cache[cache_key]
    
    # Handle timezone-aware or naive timestamps
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    if start_ts.tz is None:
        start_ts = start_ts.tz_localize('Europe/Amsterdam')
    if end_ts.tz is None:
        end_ts = end_ts.tz_localize('Europe/Amsterdam')
    
    if contract_type == 'dynamic':
        # Dynamic: spot price + margin + tax
        if resolution == '15min':
            df = spot_15m[(spot_15m['datetime'] >= start_ts) & (spot_15m['datetime'] <= end_ts)].copy()
            margin = margin_segments[margin_segments['provider'] == provider]['margin_eur_per_kwh'].iloc[0]
            year = df['datetime'].iloc[0].year if len(df) > 0 else start_ts.year
            tax = energy_tax_year[energy_tax_year['year'] == year]['energy_tax_eur_per_kwh'].iloc[0]
            df['cost_eur_per_kwh'] = df['spot_eur_per_kwh'] * (1 + VAT_RATE) + margin + tax
            result = df[['datetime', 'cost_eur_per_kwh']].copy()
        elif resolution == 'hourly':
            df = spot_hourly[(spot_hourly['datetime_hour'] >= start_ts) & (spot_hourly['datetime_hour'] <= end_ts)].copy()
            margin = margin_segments[margin_segments['provider'] == provider]['margin_eur_per_kwh'].iloc[0]
            year = df['datetime_hour'].iloc[0].year if len(df) > 0 else start_ts.year
            tax = energy_tax_year[energy_tax_year['year'] == year]['energy_tax_eur_per_kwh'].iloc[0]
            df['cost_eur_per_kwh'] = df['spot_hourly_mean_eur_per_kwh'] * (1 + VAT_RATE) + margin + tax
            result = df[['datetime_hour', 'cost_eur_per_kwh']].copy()
            result.columns = ['datetime', 'cost_eur_per_kwh']
        else:  # daily
            df = spot_daily[(spot_daily['date'] >= start_ts.date()) & (spot_daily['date'] <= end_ts.date())].copy()
            margin = margin_segments[margin_segments['provider'] == provider]['margin_eur_per_kwh'].iloc[0]
            year = pd.Timestamp(df['date'].iloc[0]).year if len(df) > 0 else start_ts.year
            tax = energy_tax_year[energy_tax_year['year'] == year]['energy_tax_eur_per_kwh'].iloc[0]
            df['cost_eur_per_kwh'] = df['spot_daily_mean_eur_per_kwh'] * (1 + VAT_RATE) + margin + tax
            df['datetime'] = pd.to_datetime(df['date'], utc=True).dt.tz_convert('Europe/Amsterdam')
            result = df[['datetime', 'cost_eur_per_kwh']].copy()
    
    else:  # fixed_variable
        if resolution == '15min':
            times = pd.date_range(start_ts, end_ts, freq='15min', tz='Europe/Amsterdam')
            costs = []
            for ts in times:
                is_peak = not is_dal_hour(ts)
                segment = fv_segments[(fv_segments['provider'] == provider) & 
                                     (fv_segments['contract_name'] == contract_name) &
                                     (fv_segments['month_start'] <= ts) & 
                                     (ts < fv_segments['month_end'])]
                if len(segment) > 0:
                    piek_price = segment.iloc[0]['piek_price_eur_per_kwh']
                    dal_price = segment.iloc[0]['dal_price_eur_per_kwh']
                    if pd.isna(dal_price):
                        dal_price = piek_price
                    price = piek_price if is_peak else dal_price
                    costs.append(price)
                else:
                    costs.append(np.nan)
            result = pd.DataFrame({'datetime': times, 'cost_eur_per_kwh': costs})
        
        elif resolution == 'hourly':
            times = pd.date_range(start_ts, end_ts, freq='1h', tz='Europe/Amsterdam')
            costs = []
            for ts in times:
                is_peak = not is_dal_hour(ts)
                segment = fv_segments[(fv_segments['provider'] == provider) & 
                                     (fv_segments['contract_name'] == contract_name) &
                                     (fv_segments['month_start'] <= ts) & 
                                     (ts < fv_segments['month_end'])]
                if len(segment) > 0:
                    piek_price = segment.iloc[0]['piek_price_eur_per_kwh']
                    dal_price = segment.iloc[0]['dal_price_eur_per_kwh']
                    if pd.isna(dal_price):
                        dal_price = piek_price
                    price = piek_price if is_peak else dal_price
                    costs.append(price)
                else:
                    costs.append(np.nan)
            result = pd.DataFrame({'datetime': times, 'cost_eur_per_kwh': costs})
        
        else:  # daily
            times = pd.date_range(start_ts, end_ts, freq='1d', tz='Europe/Amsterdam')
            costs = []
            for ts in times:
                segment = fv_segments[(fv_segments['provider'] == provider) & 
                                     (fv_segments['contract_name'] == contract_name) &
                                     (fv_segments['month_start'] <= ts) & 
                                     (ts < fv_segments['month_end'])]
                if len(segment) > 0:
                    piek = segment.iloc[0]['piek_price_eur_per_kwh']
                    dal = segment.iloc[0]['dal_price_eur_per_kwh']
                    if pd.isna(dal):
                        dal = piek
                    price = (piek + dal) / 2
                    costs.append(price)
                else:
                    costs.append(np.nan)
            result = pd.DataFrame({'datetime': times, 'cost_eur_per_kwh': costs})
    
    series_cache[cache_key] = result
    return result

print("✓ get_series() function ready (caches results in series_cache)")    

✓ get_series() function ready (caches results in series_cache)


## 6. Example Comparison: Frank Energie (Dynamic) vs Vattenfall (Variable)

In [96]:
# Helper function for dal hour detection and contract validation
def is_dal_hour(ts):
    """Check if timestamp is during DAL (night) hours"""
    hour = ts.hour
    return hour in DAL_HOURS

def find_complete_contracts_2025():
    """Find contracts with both piek and dal for all 2025 months"""
    months_2025 = df_fixed_monthly[df_fixed_monthly['month'].str.contains('2025')]['month'].unique()
    results = []
    
    for provider in df_fixed_monthly['provider'].unique():
        for contract_name in df_fixed_monthly[df_fixed_monthly['provider'] == provider]['contract_name'].unique():
            # Get all records for this provider/contract in 2025
            contracts = df_fixed_monthly[
                (df_fixed_monthly['provider'] == provider) &
                (df_fixed_monthly['contract_name'] == contract_name) &
                (df_fixed_monthly['month'].str.contains('2025'))
            ]
            
            # Check if all months have both piek and dal
            if len(contracts) == len(months_2025):  # Has all months
                if (contracts['piek_tariff'].notna().all() and contracts['dal_tariff'].notna().all()):
                    avg_piek = contracts['piek_tariff'].mean()
                    avg_dal = contracts['dal_tariff'].mean()
                    # Check that dal is actually different from piek (not fallback)
                    if avg_dal > 0 and abs(avg_dal - avg_piek) > 0.001:
                        results.append({
                            'provider': provider,
                            'contract_name': contract_name,
                            'avg_piek': avg_piek,
                            'avg_dal': avg_dal,
                            'months_count': len(contracts)
                        })
    
    return pd.DataFrame(results) if results else pd.DataFrame()

# Find suitable contracts
complete_contracts_2025 = find_complete_contracts_2025()
print("✓ Contract validation complete")
print(f"  Contracts with full piek+dal for all 2025 months: {len(complete_contracts_2025)}")
if len(complete_contracts_2025) > 0:
    print("\n  Available contracts:")
    for idx, row in complete_contracts_2025.iterrows():
        print(f"    {row['provider']}: {row['contract_name']} (piek: €{row['avg_piek']:.4f}, dal: €{row['avg_dal']:.4f})")


✓ Contract validation complete
  Contracts with full piek+dal for all 2025 months: 1

  Available contracts:
    Eneco Zakelijk: Variabel leveringstarief (piek: €0.3021, dal: €0.2675)


In [ ]:
# Time period selector with year and month toggles
period_options = {'1 Day': 1, '1 Week': 7, '1 Month': 30, '1 Year': 365}
dropdown_period = widgets.Dropdown(
    options=list(period_options.keys()),
    value='1 Month',
    description='Time Period:'
)

# Year selector
dropdown_year = widgets.Dropdown(
    options=['2025', '2026'],
    value='2025',
    description='Year:'
)

# Month selector (for 2025 when period is "1 Month")
months_2025 = ['February', 'March', 'April', 'May', 'June', 'July', 
               'August', 'September', 'October', 'November', 'December']
dropdown_month_2025 = widgets.Dropdown(
    options=months_2025,
    value='February',
    description='Month 2025:'
)

# Date picker for "1 Day" view
date_picker_day = widgets.DatePicker(
    value=pd.Timestamp('2025-02-01').date(),
    description='Select Day:',
    disabled=False
)

# Date picker for "1 Week" view (select end date of week)
date_picker_week = widgets.DatePicker(
    value=pd.Timestamp('2025-02-07').date(),
    description='Week Ending:',
    disabled=False
)

output_comparison = widgets.Output()

def update_visibility(change=None):
    """Show/hide selectors based on period and year selection"""
    period = dropdown_period.value
    year = dropdown_year.value
    
    # Hide all optional selectors first
    dropdown_month_2025.layout = widgets.Layout(display='none')
    date_picker_day.layout = widgets.Layout(display='none')
    date_picker_week.layout = widgets.Layout(display='none')
    
    # Show relevant selectors based on period
    if period == '1 Day':
        date_picker_day.layout = widgets.Layout(display='block')
    elif period == '1 Week':
        date_picker_week.layout = widgets.Layout(display='block')
    elif period == '1 Month' and year == '2025':
        dropdown_month_2025.layout = widgets.Layout(display='block')

def run_comparison(change=None):
    output_comparison.clear_output(wait=True)
    with output_comparison:
        period_label = dropdown_period.value
        selected_year = dropdown_year.value
        is_year = period_label == '1 Year'
        is_month = period_label == '1 Month'
        
        if is_year:
            # Full year view (only 2025 has complete contracts)
            if selected_year == '2025':
                start_date = pd.Timestamp('2025-02-01', tz='Europe/Amsterdam')
                end_date = pd.Timestamp('2025-12-31', tz='Europe/Amsterdam')
                resolution = 'daily'
                
                if len(complete_contracts_2025) == 0:
                    print("✗ No complete contracts found for 2025 (need both piek and dal tariffs for all months)")
                    return
                
                # Select first variable contract from the list
                selected_contract = complete_contracts_2025.iloc[0]
                provider_name = selected_contract['provider']
                contract_name = selected_contract['contract_name']
                
                print(f"📊 YEAR 2025 COMPARISON")
                print(f"Fixed/Variable Contract: {provider_name} - {contract_name}\n")
            else:
                print(f"✗ Full year {selected_year} not available (need complete contracts for all months)")
                return
        elif is_month:
            # Month view
            if selected_year == '2025':
                selected_month = dropdown_month_2025.value
                # Find contracts for this month in 2025
                month_str = f"{selected_month} 2025"
                month_contracts = df_fixed_monthly[df_fixed_monthly['month'] == month_str]
                
                if len(month_contracts) == 0:
                    print(f"✗ No contracts found for {month_str}")
                    return
                
                month_date = pd.to_datetime(month_str)
                start_date = pd.Timestamp(month_date.year, month_date.month, 1, tz='Europe/Amsterdam')
                # Go to start of next month (or end of year)
                if month_date.month == 12:
                    end_date = pd.Timestamp(month_date.year + 1, 1, 1, tz='Europe/Amsterdam')
                else:
                    end_date = pd.Timestamp(month_date.year, month_date.month + 1, 1, tz='Europe/Amsterdam')
                end_date = end_date - pd.Timedelta(days=1)
                resolution = 'hourly'
                
                print(f"📊 MONTH {month_str.upper()} COMPARISON\n")
            else:
                # 2026 or other year - use current day logic for now
                days_back = 30
                end_date = pd.Timestamp.now(tz='Europe/Amsterdam').normalize() + pd.Timedelta(days=1)
                start_date = end_date - pd.Timedelta(days=days_back)
                resolution = 'hourly'
        else:
            # Day or week view
            is_day = period_label == '1 Day'
            is_week = period_label == '1 Week'
            
            if is_day:
                # Use date picker for specific day
                selected_date = date_picker_day.value
                start_date = pd.Timestamp(selected_date, tz='Europe/Amsterdam')
                end_date = start_date + pd.Timedelta(days=1)
                resolution = '15min'
                print(f"📊 DAY VIEW - {selected_date.strftime('%A, %B %d, %Y')}\n")
                
            elif is_week:
                # Use date picker for week ending date
                week_end_date = date_picker_week.value
                start_date = pd.Timestamp(week_end_date, tz='Europe/Amsterdam') - pd.Timedelta(days=7)
                end_date = pd.Timestamp(week_end_date, tz='Europe/Amsterdam') + pd.Timedelta(days=1)
                resolution = 'hourly'
                week_start = (start_date + pd.Timedelta(days=1)).date()
                print(f"📊 WEEK VIEW - {week_start} to {week_end_date}\n")
            else:
                # Fallback for any other case
                days_back = period_options[period_label]
                end_date = pd.Timestamp.now(tz='Europe/Amsterdam').normalize() + pd.Timedelta(days=1)
                start_date = end_date - pd.Timedelta(days=days_back)
                resolution = 'hourly'
        
        # Get Frank Energie dynamic contract
        frank_energy_dynamic = get_series('dynamic', 'Frank Energie', '', start_date, end_date, resolution)
        
        if is_year and selected_year == '2025':
            # Use pre-selected contract for year
            fv_contract = get_series(
                'fixed_variable', 
                provider_name, 
                contract_name,
                start_date, 
                end_date, 
                resolution
            )
            fv_display = f"{provider_name} ({contract_name})"
            contract_found = True
        elif is_month and selected_year == '2025':
            # Use Eneco contract for this month
            month_str = f"{dropdown_month_2025.value} 2025"
            eneco_contracts = df_fixed_monthly[
                (df_fixed_monthly['provider'] == complete_contracts_2025.iloc[0]['provider']) & 
                (df_fixed_monthly['contract_name'] == complete_contracts_2025.iloc[0]['contract_name']) &
                (df_fixed_monthly['month'] == month_str)
            ]
            
            if len(eneco_contracts) > 0:
                eneco_contract = eneco_contracts.iloc[0]
                fv_contract = get_series(
                    'fixed_variable', 
                    eneco_contract['provider'], 
                    eneco_contract['contract_name'],
                    start_date, 
                    end_date, 
                    resolution
                )
                fv_display = f"{eneco_contract['provider']} - {eneco_contract['contract_name']}"
                contract_found = True
            else:
                contract_found = False
        else:
            # Use Eneco contract for day/week views (available in 2025)
            if start_date.year == 2025 and len(complete_contracts_2025) > 0:
                eneco_provider = complete_contracts_2025.iloc[0]['provider']
                eneco_contract_name = complete_contracts_2025.iloc[0]['contract_name']
                
                fv_contract = get_series(
                    'fixed_variable', 
                    eneco_provider, 
                    eneco_contract_name,
                    start_date, 
                    end_date, 
                    resolution
                )
                fv_display = f"{eneco_provider} - {eneco_contract_name}"
                contract_found = True
            else:
                # Fallback for non-2025 dates
                month_name_map = {
                    'January': 'January', 'February': 'February', 'March': 'March', 'April': 'April',
                    'May': 'May', 'June': 'June', 'July': 'July', 'August': 'August',
                    'September': 'September', 'October': 'October', 'November': 'November', 'December': 'December'
                }
                
                # Try current month and adjacent months
                months_to_try = []
                for offset in [0, -1, 1]:
                    try_date = start_date + pd.Timedelta(days=offset*15)
                    try_month = f"{month_name_map.get(try_date.strftime('%B'), try_date.strftime('%B'))} {try_date.year}"
                    months_to_try.append(try_month)
                
                vattenfall_contracts = df_fixed_monthly[
                    (df_fixed_monthly['provider'] == 'Vattenfall') & 
                    (df_fixed_monthly['month'].isin(months_to_try))
                ]
                
                if len(vattenfall_contracts) > 0:
                    vattenfall_contract = vattenfall_contracts.iloc[0]
                    fv_contract = get_series(
                        'fixed_variable', 
                        'Vattenfall', 
                        vattenfall_contract['contract_name'],
                        start_date, 
                        end_date, 
                        resolution
                    )
                    fv_display = f"Vattenfall {vattenfall_contract['contract_type']}"
                    contract_found = True
                else:
                    contract_found = False
        
        if contract_found:
            # Create comparison figure
            fig = go.Figure()
            
            if not frank_energy_dynamic.empty:
                fig.add_trace(go.Scatter(
                    x=frank_energy_dynamic['datetime'],
                    y=frank_energy_dynamic['cost_eur_per_kwh'],
                    mode='lines',
                    name='Frank Energie (Dynamic)',
                    line=dict(color='#1f77b4', width=2),
                    hovertemplate='<b>Frank Energie</b><br>Time: %{x}<br>Cost: €%{y:.4f}/kWh<extra></extra>'
                ))
            
            if not fv_contract.empty:
                fig.add_trace(go.Scatter(
                    x=fv_contract['datetime'],
                    y=fv_contract['cost_eur_per_kwh'],
                    mode='lines',
                    name=fv_display,
                    line=dict(color='#ff7f0e', width=2),
                    hovertemplate='<b>Fixed/Variable</b><br>Time: %{x}<br>Cost: €%{y:.4f}/kWh<extra></extra>'
                ))
            
            fig.update_layout(
                title=f'{period_label}: Frank Energie (Dynamic) vs {fv_display}',
                xaxis_title='Date & Time' if resolution == '15min' else 'Date',
                yaxis_title='Cost (€/kWh)',
                template='plotly_white',
                height=600,
                hovermode='x unified',
                showlegend=True,
                legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
            )
            
            fig.show()
            
            # Statistics
            print("\n" + "="*80)
            print(f"COST COMPARISON - {period_label.upper()} ({resolution.capitalize()} Resolution)")
            print(f"Period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
            print("="*80)
            
            if not frank_energy_dynamic.empty:
                print(f"\n📊 Frank Energie (Dynamic):")
                print(f"   Average:     €{frank_energy_dynamic['cost_eur_per_kwh'].mean():.4f}/kWh")
                print(f"   Min:         €{frank_energy_dynamic['cost_eur_per_kwh'].min():.4f}/kWh")
                print(f"   Max:         €{frank_energy_dynamic['cost_eur_per_kwh'].max():.4f}/kWh")
                print(f"   Std Dev:     €{frank_energy_dynamic['cost_eur_per_kwh'].std():.4f}/kWh")
                total_frank = frank_energy_dynamic['cost_eur_per_kwh'].sum()
                print(f"   Total Cost:  €{total_frank:.2f} (for 1000 kWh)")
            
            if not fv_contract.empty:
                print(f"\n📊 {fv_display}:")
                print(f"   Average:     €{fv_contract['cost_eur_per_kwh'].mean():.4f}/kWh")
                print(f"   Min:         €{fv_contract['cost_eur_per_kwh'].min():.4f}/kWh")
                print(f"   Max:         €{fv_contract['cost_eur_per_kwh'].max():.4f}/kWh")
                print(f"   Std Dev:     €{fv_contract['cost_eur_per_kwh'].std():.4f}/kWh")
                total_fv = fv_contract['cost_eur_per_kwh'].sum()
                print(f"   Total Cost:  €{total_fv:.2f} (for 1000 kWh)")
            
            # Comparison
            if not frank_energy_dynamic.empty and not fv_contract.empty:
                diff = total_frank - total_fv
                pct_diff = (diff / total_fv) * 100 if total_fv != 0 else 0
                
                print(f"\n💰 COMPARISON:")
                print(f"   Difference: €{abs(diff):.2f}")
                print(f"   Percentage: {abs(pct_diff):.1f}%")
                if diff < 0:
                    print(f"   ✓ Frank Energie is €{abs(diff):.2f} CHEAPER")
                else:
                    print(f"   ✗ Frank Energie is €{abs(diff):.2f} MORE EXPENSIVE")
        else:
            print("✗ No variable contracts found for selected period")

dropdown_period.observe(run_comparison, names='value')
dropdown_year.observe(update_visibility, names='value')
dropdown_year.observe(run_comparison, names='value')
dropdown_month_2025.observe(run_comparison, names='value')
dropdown_period.observe(update_visibility, names='value')
date_picker_day.observe(run_comparison, names='value')
date_picker_week.observe(run_comparison, names='value')

# Display selector and run initial comparison
print("\n📊 EXAMPLE COMPARISON: Frank Energie (Dynamic) vs Fixed/Variable Contracts")
print("="*80)
print("Select time period and year:\n")

# Create control panel with year, period, and optional selectors
controls_display = widgets.VBox([
    widgets.HBox([dropdown_year, dropdown_period]),
    date_picker_day,
    date_picker_week,
    dropdown_month_2025
])

display(controls_display)
display(output_comparison)

print("\nGenerating initial comparison...\n")
update_visibility()
run_comparison()


📊 EXAMPLE COMPARISON: Frank Energie (Dynamic) vs Fixed/Variable Contracts
Select time period and year:



Output()


Generating initial comparison...

